<a href="https://colab.research.google.com/github/cristianzucconi2-web/iatoarts/blob/main/iatoarts3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- 1. SETUP LIBRERIE ---
!pip install mido
import random
import mido
from mido import Message, MidiFile, MidiTrack

# --- 2. CONFIGURAZIONE MUSICALE ---
# Scala di Do Minore (Epica/Cinematografica)
SCALA_PIANO = [60, 62, 63, 65, 67, 68, 70, 72, 74, 75]
SCALA_BASSO = [36, 39, 43, 48] # Solo le note portanti della scala

DURATA_BLOCCO = 32      # Note per ogni strumento in un blocco
NUMERO_BLOCCHI = 6      # 6 blocchi da ~15-20 sec = quasi 2 minuti
POPOLAZIONE_SIZE = 80   # Più tentativi per generazione
GENERAZIONI = 100       # Quante volte l'IA "studia" ogni blocco

# --- 3. FUNZIONE DI FITNESS (IL GIUDICE) ---
def calcola_fitness(individuo):
    piano = individuo[0]
    basso = individuo[1]
    voto = 0

    for i in range(DURATA_BLOCCO):
        # A. Regola Tonalità: Le note devono essere nella scala
        if piano[i] not in SCALA_PIANO: voto -= 100
        if basso[i] not in SCALA_BASSO: voto -= 100

        # B. Regola Armonia: Il piano e il basso devono "andare d'accordo" (intervalli consonanti)
        if abs(piano[i] - basso[i]) % 12 in [0, 3, 7]: voto += 50

        # C. Regola Fluidità: No salti da elefante al piano
        if i < DURATA_BLOCCO - 1:
            dist = abs(piano[i] - piano[i+1])
            if 1 <= dist <= 2: voto += 40  # Passaggi fluidi
            if dist > 7: voto -= 80        # Penalità salti enormi
            if dist == 0: voto -= 10       # Penalità se suona troppe volte la stessa nota (noia)

        # D. Regola Basso: Deve essere stabile
        if i < DURATA_BLOCCO - 1:
            if basso[i] == basso[i+1]: voto += 30 # Il basso che tiene la nota è buono

    return voto

# --- 4. MOTORE EVOLUTIVO ---
def genera_blocco_ottimizzato():
    # Creiamo popolazione iniziale
    popolazione = []
    for _ in range(POPOLAZIONE_SIZE):
        p = [random.choice(SCALA_PIANO) for _ in range(DURATA_BLOCCO)]
        b = [random.choice(SCALA_BASSO) for _ in range(DURATA_BLOCCO)]
        popolazione.append([p, b])

    for g in range(GENERAZIONI):
        # Ordiniamo per voto
        popolazione = sorted(popolazione, key=calcola_fitness, reverse=True)
        migliori = popolazione[:15]
        nuova_gen = migliori.copy()

        while len(nuova_gen) < POPOLAZIONE_SIZE:
            # Crossover (Incrocio)
            p1, p2 = random.choice(migliori), random.choice(migliori)
            taglio = random.randint(1, DURATA_BLOCCO - 1)
            figlio = [p1[0][:taglio] + p2[0][taglio:], p1[1][:taglio] + p2[1][taglio:]]

            # Mutazione (Cambio nota casuale)
            if random.random() < 0.2:
                figlio[0][random.randint(0, DURATA_BLOCCO-1)] = random.choice(SCALA_PIANO)

            nuova_gen.append(figlio)
        popolazione = nuova_gen

    return popolazione[0] # Restituiamo il vincitore del blocco

# --- 5. COMPOSIZIONE FINALE ---
canzone_piano, canzone_basso = [], []

print("Composizione in corso (6 blocchi)...")
for b in range(NUMERO_BLOCCHI):
    vincitore = genera_blocco_ottimizzato()
    canzone_piano.extend(vincitore[0])
    canzone_basso.extend(vincitore[1])
    print(f"Blocco {b+1} completato!")

# --- 6. SALVATAGGIO MIDI MULTITRACCIA ---
mid = MidiFile()
t_piano = MidiTrack()
t_basso = MidiTrack()
t_drums = MidiTrack()
mid.tracks.extend([t_piano, t_basso, t_drums])

# Assegnazione nomi e strumenti
t_piano.append(mido.MetaMessage('track_name', name='Piano AI', time=0))
t_basso.append(mido.MetaMessage('track_name', name='Basso AI', time=0))
t_drums.append(mido.MetaMessage('track_name', name='Drums AI', time=0))

for i in range(len(canzone_piano)):
    # Piano
    t_piano.append(Message('note_on', note=canzone_piano[i], velocity=85, time=0))
    t_piano.append(Message('note_off', note=canzone_piano[i], velocity=85, time=480))
    # Basso
    t_basso.append(Message('note_on', note=canzone_basso[i], velocity=100, time=0))
    t_basso.append(Message('note_off', note=canzone_basso[i], velocity=100, time=480))
    # Batteria (Canale 9, Note 36=Cassa, 38=Rullante)
    kick_snare = 36 if i % 2 == 0 else 38
    t_drums.append(Message('note_on', note=kick_snare, velocity=95, time=0, channel=9))
    t_drums.append(Message('note_off', note=kick_snare, velocity=95, time=480, channel=9))

mid.save('composizione_finale_ai.mid')
print("\nFatto! Scarica 'composizione_finale_ai.mid' e mettilo su GarageBand.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.0 MB/s eta 0:00:00
Composizione in corso (6 blocchi)...
Blocco 1 completato!
Blocco 2 completato!
Blocco 3 completato!
Blocco 4 completato!
Blocco 5 completato!
Blocco 6 completato!

Fatto! Scarica 'composizione_finale_ai.mid' e mettilo su GarageBand.
